# 03 — External Context Dataset Generation

This notebook generates the **External Context Dataset** (`data/raw/external_context.csv`)  
containing calendar, holiday, weather, and event features for every **date × city** combination.

**Output schema:**

| Column | Type | Description |
|---|---|---|
| `date` | date | YYYY-MM-DD |
| `city` | str | City name |
| `day_of_week` | int | 0=Mon … 6=Sun |
| `is_weekend` | bool | Sat/Sun flag |
| `is_holiday` | bool | National holiday flag |
| `month` | int | 1–12 |
| `weather` | str | Clear / Cloudy / Rainy / Stormy / Snowy / Unknown |
| `event_flag` | bool | Festival / sports / local event flag |

## 1. Setup & Imports

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Paths ────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"

RESTAURANT_CSV = RAW_DIR / "restaurant_dataset.csv"
OUTPUT_CSV = RAW_DIR / "external_context.csv"

print(f"Project root : {PROJECT_ROOT}")
print(f"Restaurant CSV: {RESTAURANT_CSV}")
print(f"Output CSV   : {OUTPUT_CSV}")

Project root : C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj
Restaurant CSV: C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj\data\raw\restaurant_dataset.csv
Output CSV   : C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj\data\raw\external_context.csv


## 2. Load Restaurant Data & Extract Cities

In [2]:
restaurants = pd.read_csv(RESTAURANT_CSV)
cities = sorted(restaurants["City"].dropna().unique().tolist())

print(f"Total restaurants: {len(restaurants)}")
print(f"Unique cities   : {len(cities)}")
print(f"\nTop 10 cities by restaurant count:")
print(restaurants["City"].value_counts().head(10).to_string())

Total restaurants: 9551
Unique cities   : 141

Top 10 cities by restaurant count:
City
New Delhi       5473
Gurgaon         1118
Noida           1080
Faridabad        251
Ghaziabad         25
Bhubaneshwar      21
Lucknow           21
Ahmedabad         21
Amritsar          21
Guwahati          21


## 3. Define Date Range

Two full years: **2024-01-01 → 2025-12-31** (730 days).  
If a `daily_orders.csv` already exists we automatically match its date range instead.

In [3]:
daily_orders_path = RAW_DIR / "daily_orders.csv"

if daily_orders_path.exists():
    orders_df = pd.read_csv(daily_orders_path, parse_dates=["date"])
    START_DATE = orders_df["date"].min()
    END_DATE = orders_df["date"].max()
    print(f"Matched daily_orders.csv date range: {START_DATE.date()} → {END_DATE.date()}")
else:
    START_DATE = pd.Timestamp("2024-01-01")
    END_DATE = pd.Timestamp("2025-12-31")
    print(f"No daily_orders.csv found — using default range: {START_DATE.date()} → {END_DATE.date()}")

date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="D")
print(f"Total days: {len(date_range)}")

No daily_orders.csv found — using default range: 2024-01-01 → 2025-12-31
Total days: 731


## 4. Generate Date × City Cross Product

In [4]:
# Build the cross product of every date with every city
rows = []
for dt in date_range:
    for city in cities:
        rows.append({"date": dt, "city": city})

df = pd.DataFrame(rows)
print(f"Cross product shape: {df.shape}  ({len(date_range)} dates × {len(cities)} cities)")

Cross product shape: (103071, 2)  (731 dates × 141 cities)


## 5. Calendar Features (Deterministic)

In [5]:
df["day_of_week"] = df["date"].dt.dayofweek          # 0=Monday … 6=Sunday
df["is_weekend"]  = df["day_of_week"] >= 5            # Saturday=5, Sunday=6
df["month"]       = df["date"].dt.month               # 1–12

# Quick sanity check
print("Day of week distribution:")
print(df["day_of_week"].value_counts().sort_index().to_string())
print(f"\nWeekend %: {df['is_weekend'].mean() * 100:.1f}%")

Day of week distribution:
day_of_week
0    14805
1    14805
2    14805
3    14664
4    14664
5    14664
6    14664

Weekend %: 28.5%


## 6. Holiday Flags

We define major **Indian national holidays** (the primary market — ~85 % of restaurants)  
plus widely observed international holidays. Multi-day festivals are expanded into date ranges.

In [6]:
# ── Indian & International holidays (both 2024 and 2025) ───
HOLIDAYS_RAW = {
    # ── 2024 ──────────────────────────────────────────────────
    # Republic Day
    "2024-01-26": "Republic Day",
    # Holi
    "2024-03-25": "Holi",
    # Good Friday
    "2024-03-29": "Good Friday",
    # Eid-ul-Fitr (approx)
    "2024-04-11": "Eid-ul-Fitr",
    "2024-04-12": "Eid-ul-Fitr",
    # Labour Day
    "2024-05-01": "Labour Day",
    # Eid-ul-Adha (approx)
    "2024-06-17": "Eid-ul-Adha",
    "2024-06-18": "Eid-ul-Adha",
    # Independence Day
    "2024-08-15": "Independence Day",
    # Janmashtami
    "2024-08-26": "Janmashtami",
    # Gandhi Jayanti
    "2024-10-02": "Gandhi Jayanti",
    # Navratri (9 days)
    **{f"2024-10-{d:02d}": "Navratri" for d in range(3, 12)},
    # Dussehra
    "2024-10-12": "Dussehra",
    # Diwali (5 days)
    **{f"2024-11-{d:02d}": "Diwali" for d in range(1, 6)},
    # Christmas
    "2024-12-25": "Christmas",
    # New Year's Eve
    "2024-12-31": "New Year's Eve",

    # ── 2025 ──────────────────────────────────────────────────
    # New Year
    "2025-01-01": "New Year",
    # Republic Day
    "2025-01-26": "Republic Day",
    # Holi
    "2025-03-14": "Holi",
    # Good Friday
    "2025-04-18": "Good Friday",
    # Eid-ul-Fitr (approx)
    "2025-03-31": "Eid-ul-Fitr",
    "2025-04-01": "Eid-ul-Fitr",
    # Labour Day
    "2025-05-01": "Labour Day",
    # Eid-ul-Adha (approx)
    "2025-06-07": "Eid-ul-Adha",
    "2025-06-08": "Eid-ul-Adha",
    # Independence Day
    "2025-08-15": "Independence Day",
    # Janmashtami
    "2025-08-16": "Janmashtami",
    # Gandhi Jayanti
    "2025-10-02": "Gandhi Jayanti",
    # Navratri (9 days)
    **{f"2025-09-{d:02d}": "Navratri" for d in range(22, 31)},
    "2025-10-01": "Navratri",
    # Dussehra
    "2025-10-02": "Dussehra",
    # Diwali (5 days)
    **{f"2025-10-{d:02d}": "Diwali" for d in range(20, 25)},
    # Christmas
    "2025-12-25": "Christmas",
    # New Year's Eve
    "2025-12-31": "New Year's Eve",
}

holiday_dates = set(pd.to_datetime(list(HOLIDAYS_RAW.keys())))

df["is_holiday"] = df["date"].isin(holiday_dates)

print(f"Total holiday dates defined: {len(holiday_dates)}")
print(f"Holiday rows flagged: {df['is_holiday'].sum():,} "
      f"({df['is_holiday'].mean() * 100:.2f}%)")

# Show which holidays we flagged
flagged = df[df["is_holiday"]][["date"]].drop_duplicates().sort_values("date")
flagged["holiday_name"] = flagged["date"].dt.strftime("%Y-%m-%d").map(HOLIDAYS_RAW)
print("\nHoliday calendar:")
print(flagged.to_string(index=False))

Total holiday dates defined: 57
Holiday rows flagged: 8,037 (7.80%)

Holiday calendar:
      date     holiday_name
2024-01-26     Republic Day
2024-03-25             Holi
2024-03-29      Good Friday
2024-04-11      Eid-ul-Fitr
2024-04-12      Eid-ul-Fitr
2024-05-01       Labour Day
2024-06-17      Eid-ul-Adha
2024-06-18      Eid-ul-Adha
2024-08-15 Independence Day
2024-08-26      Janmashtami
2024-10-02   Gandhi Jayanti
2024-10-03         Navratri
2024-10-04         Navratri
2024-10-05         Navratri
2024-10-06         Navratri
2024-10-07         Navratri
2024-10-08         Navratri
2024-10-09         Navratri
2024-10-10         Navratri
2024-10-11         Navratri
2024-10-12         Dussehra
2024-11-01           Diwali
2024-11-02           Diwali
2024-11-03           Diwali
2024-11-04           Diwali
2024-11-05           Diwali
2024-12-25        Christmas
2024-12-31   New Year's Eve
2025-01-01         New Year
2025-01-26     Republic Day
2025-03-14             Holi
2025-03-31      E

## 7. Weather Conditions (Synthetic – Seasonal)

Weather is generated synthetically with **season-aware probability distributions**:  
- **Monsoon (Jun–Sep):** Higher probability of Rainy / Stormy  
- **Winter (Dec–Feb):** Some probability of Snowy for northern cities  
- **Summer / Other:** Mostly Clear / Cloudy

In [7]:
# Cities in northern India that can see snow / extreme cold
NORTHERN_CITIES = {
    "New Delhi", "Gurgaon", "Noida", "Faridabad", "Ghaziabad",
    "Lucknow", "Amritsar", "Chandigarh", "Jaipur",
}

# Tropical / coastal cities with distinct monsoon
COASTAL_CITIES = {
    "Mumbai", "Goa", "Chennai", "Kochi", "Mangalore",
    "Kolkata", "Vizag", "Visakhapatnam",
}

WEATHER_OPTIONS = ["Clear", "Cloudy", "Rainy", "Stormy", "Snowy", "Unknown"]


def get_weather_probs(month: int, city: str) -> list:
    """Return probability distribution over WEATHER_OPTIONS."""
    # Indices: Clear, Cloudy, Rainy, Stormy, Snowy, Unknown

    is_north = city in NORTHERN_CITIES
    is_coastal = city in COASTAL_CITIES

    if month in (6, 7, 8, 9):  # Monsoon
        if is_coastal:
            return [0.10, 0.20, 0.45, 0.15, 0.00, 0.10]
        else:
            return [0.15, 0.25, 0.40, 0.10, 0.00, 0.10]

    elif month in (12, 1, 2):  # Winter
        if is_north:
            return [0.30, 0.35, 0.10, 0.02, 0.15, 0.08]
        else:
            return [0.45, 0.30, 0.10, 0.02, 0.00, 0.13]

    elif month in (3, 4, 5):  # Summer
        return [0.50, 0.25, 0.08, 0.05, 0.00, 0.12]

    else:  # Oct, Nov – post-monsoon
        return [0.40, 0.30, 0.15, 0.05, 0.00, 0.10]


# Vectorised generation
weather_labels = []
for month, city in zip(df["month"], df["city"]):
    probs = get_weather_probs(month, city)
    weather_labels.append(np.random.choice(WEATHER_OPTIONS, p=probs))

df["weather"] = weather_labels

print("Weather distribution:")
print(df["weather"].value_counts(normalize=True).mul(100).round(1).to_string())

Weather distribution:
weather
Clear      35.2
Cloudy     27.1
Rainy      20.4
Unknown    11.0
Stormy      6.1
Snowy       0.3


## 8. Event Flags

Events include:
- **Major festivals:** Diwali week, Holi, Navratri, Christmas week, New Year week  
- **Sports:** IPL season (late March – late May)  
- **Random local events:** ~3% probability on non-holiday / non-festival days

In [8]:
# ── Festival / event date ranges ─────────────────────────────
EVENT_RANGES = [
    # 2024 events
    ("2024-01-01", "2024-01-02"),   # New Year celebrations
    ("2024-03-24", "2024-03-26"),   # Holi weekend
    ("2024-04-11", "2024-04-12"),   # Eid
    ("2024-10-03", "2024-10-12"),   # Navratri → Dussehra
    ("2024-10-29", "2024-11-05"),   # Diwali week
    ("2024-12-24", "2024-12-31"),   # Christmas → NYE
    ("2024-03-22", "2024-05-26"),   # IPL 2024 season

    # 2025 events
    ("2025-01-01", "2025-01-02"),   # New Year celebrations
    ("2025-03-13", "2025-03-15"),   # Holi weekend
    ("2025-03-31", "2025-04-01"),   # Eid
    ("2025-09-22", "2025-10-02"),   # Navratri → Dussehra
    ("2025-10-18", "2025-10-24"),   # Diwali week
    ("2025-12-24", "2025-12-31"),   # Christmas → NYE
    ("2025-03-14", "2025-05-25"),   # IPL 2025 season
]

event_dates = set()
for start, end in EVENT_RANGES:
    rng = pd.date_range(start=start, end=end, freq="D")
    event_dates.update(rng)

# Start with known events
df["event_flag"] = df["date"].isin(event_dates)

# Add random local events (~3% on non-event days)
non_event_mask = ~df["event_flag"]
random_events = np.random.rand(non_event_mask.sum()) < 0.03
df.loc[non_event_mask, "event_flag"] = random_events

print(f"Total event-flagged rows: {df['event_flag'].sum():,} "
      f"({df['event_flag'].mean() * 100:.2f}%)")

Total event-flagged rows: 29,833 (28.94%)


## 9. Validation

Run consistency checks and display summary statistics.

In [9]:
# ── Schema checks ────────────────────────────────────────────
assert df.shape[1] == 8, f"Expected 8 columns, got {df.shape[1]}"
assert df.isnull().sum().sum() == 0, "Null values found!"
assert df["day_of_week"].between(0, 6).all(), "day_of_week out of range"
assert df["month"].between(1, 12).all(), "month out of range"
assert set(df["weather"].unique()).issubset(set(WEATHER_OPTIONS)), "Unknown weather label"

# ── Spot-check: 2024-01-26 (Republic Day) should be a holiday ─
rd = df[(df["date"] == "2024-01-26")]
assert rd["is_holiday"].all(), "Republic Day not flagged as holiday!"

# ── Spot-check: day_of_week correctness ─────────────────────
sample_row = df.iloc[0]
assert sample_row["day_of_week"] == sample_row["date"].dayofweek, "day_of_week mismatch"

# ── Summary ──────────────────────────────────────────────────
print("="  * 60)
print("       EXTERNAL CONTEXT DATASET — SUMMARY")
print("=" * 60)
print(f"Shape        : {df.shape}")
print(f"Date range   : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique cities: {df['city'].nunique()}")
print(f"Weekend %    : {df['is_weekend'].mean() * 100:.1f}%")
print(f"Holiday %    : {df['is_holiday'].mean() * 100:.2f}%")
print(f"Event %      : {df['event_flag'].mean() * 100:.2f}%")
print(f"Null values  : {df.isnull().sum().sum()}")
print("\nWeather distribution:")
print(df["weather"].value_counts(normalize=True).mul(100).round(1).to_string())
print("\nDtypes:")
print(df.dtypes.to_string())
print("\n✅ All validation checks passed!")

       EXTERNAL CONTEXT DATASET — SUMMARY
Shape        : (103071, 8)
Date range   : 2024-01-01 → 2025-12-31
Unique cities: 141
Weekend %    : 28.5%
Holiday %    : 7.80%
Event %      : 28.94%
Null values  : 0

Weather distribution:
weather
Clear      35.2
Cloudy     27.1
Rainy      20.4
Unknown    11.0
Stormy      6.1
Snowy       0.3

Dtypes:
date           datetime64[ns]
city                   object
day_of_week             int32
is_weekend               bool
month                   int32
is_holiday               bool
weather                object
event_flag               bool

✅ All validation checks passed!


## 10. Save to CSV

In [10]:
# Format date as YYYY-MM-DD string for clean CSV output
df["date"] = df["date"].dt.strftime("%Y-%m-%d")

# Save
df.to_csv(OUTPUT_CSV, index=False)

file_size_mb = OUTPUT_CSV.stat().st_size / (1024 * 1024)
print(f"✅ Saved to: {OUTPUT_CSV}")
print(f"   File size: {file_size_mb:.2f} MB")
print(f"   Rows     : {len(df):,}")
print(f"   Columns  : {list(df.columns)}")

✅ Saved to: C:\Users\welcome\Desktop\restaurant-sales-forecasting-system\minorProj\data\raw\external_context.csv
   File size: 4.90 MB
   Rows     : 103,071
   Columns  : ['date', 'city', 'day_of_week', 'is_weekend', 'month', 'is_holiday', 'weather', 'event_flag']


In [11]:
# ── Preview the saved file ───────────────────────────────────
preview = pd.read_csv(OUTPUT_CSV)
print("Preview (first 10 rows):")
preview.head(10)

Preview (first 10 rows):


,date,city,day_of_week,is_weekend,month,is_holiday,weather,event_flag
0,2024-01-01,Abu Dhabi,0,False,1,False,Clear,True
1,2024-01-01,Agra,0,False,1,False,Unknown,True
2,2024-01-01,Ahmedabad,0,False,1,False,Cloudy,True
3,2024-01-01,Albany,0,False,1,False,Cloudy,True
4,2024-01-01,Allahabad,0,False,1,False,Clear,True
5,2024-01-01,Amritsar,0,False,1,False,Clear,True
6,2024-01-01,Ankara,0,False,1,False,Clear,True
7,2024-01-01,Armidale,0,False,1,False,Stormy,True
8,2024-01-01,Athens,0,False,1,False,Cloudy,True
9,2024-01-01,Auckland,0,False,1,False,Cloudy,True
